In [6]:
# ========================================================
# PCB DEFECT DETECTION: RESPONSE-BASED KNOWLEDGE DISTILLATION
# Student Model: YOLOv8n | Teacher: YOLOv8l (Trained)
# ========================================================

# 0. INSTALL & SETUP
# =========================================================
!pip install ultralytics --quiet

import os
import torch
from ultralytics import YOLO
from pathlib import Path
from google.colab import drive
import shutil
from datetime import datetime

# Mount Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Device
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# ========================================================
# 1. PATHS (UPDATE ONLY THESE)
# ========================================================
TEACHER_WEIGHTS = "/content/drive/MyDrive/PCB_Training/yolov8l_pcb_teacher_v8_compatible/weights/best.pt"
DATA_YAML = "/content/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/PCB_Training"
STUDENT_RUN_NAME = "yolov8n_pcb_student_responseKD"

# Verify paths
assert os.path.exists(TEACHER_WEIGHTS), f"Teacher weights not found: {TEACHER_WEIGHTS}"
assert os.path.exists(DATA_YAML), f"data.yaml not found: {DATA_YAML}"



Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: 0


In [4]:
# ============================================================
# EXTRACT ZIP FILE IN GOOGLE COLAB
# ============================================================

import zipfile
import os
from pathlib import Path
# ============================================================
# METHOD 3: Extract with Progress Bar (Advanced)
# ============================================================
print("\n\n" + "="*60)
print("METHOD 3: Extract with Progress Bar (Advanced)")
print("="*60)
print("\n# Uncomment the code below for detailed extraction progress:\n")


# Uncomment for extraction with progress bar:

from tqdm import tqdm
import zipfile

# Path to zip (update this)
ZIP_PATH = "/content/drive/MyDrive/PCB_Training/yolo_dataset_1024.zip"

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        members = zip_ref.namelist()

        print(f"📦 Extracting {len(members)} files...")
        for member in tqdm(members, desc="Extracting"):
            zip_ref.extract(member, '/content/')

    print("✅ Done!")
else:
    print(f"❌ Zip not found: {ZIP_PATH}")



# ============================================================
# HELPER: List Drive Contents
# ============================================================
print("\n\n" + "="*60)
print("HELPER: Show Google Drive Contents")
print("="*60)

def show_drive_contents(path="/content/drive/MyDrive"):
    """Show all folders and files in Google Drive"""
    print(f"\n📂 Contents of: {path}")
    try:
        p = Path(path)
        if p.exists():
            items = sorted(p.iterdir())
            for item in items[:20]:  # Show first 20 items
                if item.is_dir():
                    print(f"   📁 {item.name}/")
                else:
                    size = item.stat().st_size / (1024*1024)  # MB
                    print(f"   📄 {item.name} ({size:.2f} MB)")

            if len(items) > 20:
                print(f"   ... and {len(items) - 20} more items")
        else:
            print(f"   ❌ Path doesn't exist")
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Show MyDrive root
show_drive_contents("/content/drive/MyDrive")

# Show PCB_Dataset folder (if it exists)
show_drive_contents("/content/drive/MyDrive/PCB_Training")


# ============================================================
# FINAL CHECK
# ============================================================
print("\n\n" + "="*60)
print("FINAL CHECK")
print("="*60)

dataset_path = Path('/content/yolo_dataset')
if dataset_path.exists():
    print("✅ Dataset is ready at: /content/yolo_dataset")

    # Count files
    total_images = 0
    total_labels = 0

    for split in ['train', 'val', 'test']:
        img_path = dataset_path / split / 'images'
        lbl_path = dataset_path / split / 'labels'

        if img_path.exists():
            imgs = len(list(img_path.glob('*.jpg'))) + len(list(img_path.glob('*.png')))
            total_images += imgs

        if lbl_path.exists():
            lbls = len(list(lbl_path.glob('*.txt')))
            total_labels += lbls

    print(f"   📊 Total: {total_images} images, {total_labels} labels")
    print("\n🎉 Ready to proceed with training!")

else:
    print("❌ Dataset not extracted yet")
    print("\n💡 Next steps:")
    print("   1. Make sure yolo_dataset.zip is in Google Drive")
    print("   2. Update ZIP_PATH in METHOD 1 above")
    print("   3. Run this cell again")
    print("   OR")
    print("   4. Use METHOD 2 (direct upload)")



METHOD 3: Extract with Progress Bar (Advanced)

# Uncomment the code below for detailed extraction progress:

📦 Extracting 16633 files...


Extracting: 100%|██████████| 16633/16633 [01:00<00:00, 275.03it/s]

✅ Done!


HELPER: Show Google Drive Contents

📂 Contents of: /content/drive/MyDrive
   📄 1.jpg (0.04 MB)
   📄 5S certificate-20 Feb 2021.pdf (0.51 MB)
   📄 AdvancedDashboard.jpeg (0.19 MB)
   📄 Assignment # 2.pdf (0.31 MB)
   📄 CCN Presentation.gslides (0.00 MB)
   📄 CHINA PAKISTAN ECONOMIC CORRIDOR(CPEC).docx (0.01 MB)
   📁 Colab Notebooks/
   📄 Conditionals & Control Structures.gslides (0.00 MB)
   📄 Copy of PIAIC Q1 Video Checklist.gsheet (0.00 MB)
   📄 Course Feedback Form (Responses).gsheet (0.00 MB)
   📄 Course Feedback Form.gform (0.00 MB)
   📄 ENROLMENT_FORM_Latest.pdf (0.13 MB)
   📄 Engr. Tooba Ahmed Alvi (Resume) (1).pdf (0.11 MB)
   📄 Engr. Tooba Ahmed Alvi (Resume).pdf (0.11 MB)
   📄 Engr. Tooba Ahmed Alvi-CV.pdf (0.17 MB)
   📄 Engr. Tooba Ahmed Alvi_CV.pdf (0.10 MB)
   📄 Engr.Tooba Ahmed Alvi-CV.pdf (0.11 MB)
   📄 Engr.Tooba_Ahmed_Alvi-CV.pdf (0.16 MB)
   📄 Engr_Tooba_Ahmed_Alvi_(Resume).pdf (0.11 MB)
   📄 Figure 5 - Continental Compliance Matrix (0.00 MB)
   ... and 52 mo

In [5]:
# ============================================================
# 3. Create data.yaml file
# ============================================================
print("\n" + "="*60)
print("📄 CREATING data.yaml")
print("="*60)

YAML_FILE_PATH = '/content/data.yaml'

# Define the YAML content with correct structure and absolute paths
data_yaml_content = """
names:
- missing_hole
- mouse_bite
- open_circuit
- short
- spur
- spurious_copper
nc: 6
scale:
- 1.0
- 1.0
- 1.0
- 1.0
- 2.0
- 1.0
train: /content//train/images
val: /content//val/images
test: /content/test/images
"""

# Write the content to the file
if Path('/content/').exists():
    with open(YAML_FILE_PATH, 'w') as f:
        f.write(data_yaml_content)

    print(f"✅ data.yaml created at: {YAML_FILE_PATH}")
    print("\nContents:")
    print("------------------------------------------------------------")
    print(data_yaml_content.strip())
    print("--------------------------------------")
else:
    print(f"❌ Cannot create data.yaml. Base dataset folder '/content/yolo_dataset' not found.")
    print("💡 Did the zip file extract successfully?")




📄 CREATING data.yaml
✅ data.yaml created at: /content/data.yaml

Contents:
------------------------------------------------------------
names:
- missing_hole
- mouse_bite
- open_circuit
- short
- spur
- spurious_copper
nc: 6
scale:
- 1.0
- 1.0
- 1.0
- 1.0
- 2.0
- 1.0
train: /content//train/images
val: /content//val/images
test: /content/test/images
--------------------------------------


In [9]:
# ========================================================
# 2. LOAD TEACHER (FROZEN)
# ========================================================
print("\nLoading Teacher Model (YOLOv8l)...")
teacher = YOLO(TEACHER_WEIGHTS)
teacher.model.eval()
for p in teacher.model.parameters():
    p.requires_grad = False
print("Teacher loaded and frozen.")

# ========================================================
# 3. CUSTOM RESPONSE KD TRAINER
# ========================================================
from ultralytics.engine.trainer import BaseTrainer
from ultralytics.models.yolo.detect import DetectionTrainer
import torch.nn.functional as F

class ResponseKDTrainer(DetectionTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher.model  # Frozen teacher
        self.temperature = 4.0
        self.alpha = 0.7
        print(f"Response KD Trainer initialized: T={self.temperature}, α={self.alpha}")

    def get_model(self, cfg=None, weights=None, verbose=True):
        # Load student model
        model = super().get_model(cfg, weights, verbose)
        return model

    def loss(self, x, *args, **kwargs):
        # Get student prediction and original task loss
        student_pred = self.model(x)
        task_loss, task_loss_items = super().loss(student_pred, *args, **kwargs)

        # Get teacher prediction (no grad)
        with torch.no_grad():
            teacher_pred = self.teacher(x)

        # Response KD: Distill logits (classification head)
        # Extract class logits from predictions (assuming YOLO format: [pred, proto] or similar)
        # For simplicity, use the classification component of the output
        student_logits = student_pred[0][..., 4:]  # Shape: [bs, anchors, classes] - adjust if needed
        teacher_logits = teacher_pred[0][..., 4:]

        # Soft targets
        soft_teacher = F.log_softmax(teacher_logits / self.temperature, dim=-1)
        soft_student = F.softmax(student_logits / self.temperature, dim=-1)

        # KL divergence (response loss)
        kd_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (self.temperature ** 2)

        # Combined loss
        total_loss = task_loss + self.alpha * kd_loss
        loss_items = torch.cat((task_loss_items, torch.tensor([kd_loss.item()], device=task_loss_items.device)))
        return total_loss, loss_items

    def setup_model(self):
        super().setup_model()
        # Ensure student is on device
        self.model.to(self.device)

# ========================================================
# 4. TRAIN STUDENT WITH CUSTOM KD TRAINER
# ========================================================
print("\nStarting Response-Based KD Training (YOLOv8n)...")

student = YOLO("yolov8n.pt")  # Small & fast student

# Override trainer class for KD
student.trainer = ResponseKDTrainer

results = student.train(
    data=DATA_YAML,
    epochs=30,  # Reduced for testing; increase to 60 for full
    imgsz=512,
    batch=64,                    # Larger batch OK for small model
    device=device,
    project=PROJECT_DIR,
    name=STUDENT_RUN_NAME,
    exist_ok=True,
    patience=20,
    optimizer="AdamW",
    lr0=0.002,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cache="ram",
    amp=True,
    plots=True,
    save_period=10,

    # === AUGMENTATIONS (Same as Teacher) ===
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5, shear=2.0,
    fliplr=0.5, mosaic=1.0, mixup=0.15, copy_paste=0.1,
)

print("RESPONSE-BASED KD TRAINING COMPLETED!")


Loading Teacher Model (YOLOv8l)...
Teacher loaded and frozen.

Starting Response-Based KD Training (YOLOv8n)...
Ultralytics 8.3.232 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_pcb_student_responseKD, nbs=64

In [10]:
# ========================================================
# 4. LOAD BEST STUDENT & EVALUATE
# ========================================================
best_student_path = f"{PROJECT_DIR}/{STUDENT_RUN_NAME}/weights/best.pt"
student = YOLO(best_student_path)

print("\nEvaluating on VALIDATION SET...")
val_metrics = student.val(data=DATA_YAML, split='val', plots=True, save_json=True)

print("\nEvaluating on TEST SET...")
test_metrics = student.val(data=DATA_YAML, split='test', plots=True, save_json=True)

# ========================================================
# 5. PRINT CLASS-WISE RESULTS (BEAUTIFUL FORMAT)
# ========================================================
def print_metrics(metrics, title):
    print(f"\n{title}")
    print("=" * 90)
    print(f"{'Class':<18} {'P':>8} {'R':>8} {'mAP50':>8} {'mAP50-95':>10}")
    print("-" * 90)
    for i, name in metrics.names.items():
        print(f"{name:<18} {metrics.box.p[i]:8.4f} {metrics.box.r[i]:8.4f} {metrics.box.ap50[i]:8.4f} {metrics.box.ap[i]:10.4f}")
    print("-" * 90)
    print(f"{'OVERALL':<18} {metrics.box.mp:8.4f} {metrics.box.mr:8.4f} {metrics.box.map50:8.4f} {metrics.box.map:10.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f} | mAP50: {metrics.box.map50:.4f} | Precision: {metrics.box.mp:.4f} | Recall: {metrics.box.mr:.4f}")

print_metrics(val_metrics, "VALIDATION SET - RESPONSE KD STUDENT")
print_metrics(test_metrics, "TEST SET - RESPONSE KD STUDENT")




Evaluating on VALIDATION SET...
Ultralytics 8.3.232 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 84.1±29.0 MB/s, size: 325.9 KB)
val: Scanning /content/val/labels.cache... 1123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1123/1123 1.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 71/71 4.8it/s 14.9s
                   all       1123       4752      0.966      0.908      0.948      0.572
          missing_hole        187        810      0.985      0.991      0.993      0.698
            mouse_bite        186        787      0.958      0.865      0.912      0.526
          open_circuit        188        763      0.956      0.852      0.932      0.517
                 short        188        797      0.978      0.956      0.979      0.612
                  spur   

In [11]:
# ========================================================
# 6. SAVE FINAL RESULTS
# ========================================================
final_folder = f"/content/drive/MyDrive/PCB_ResponseKD_Final_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(final_folder, exist_ok=True)
shutil.copytree(f"{PROJECT_DIR}/{STUDENT_RUN_NAME}", final_folder, dirs_exist_ok=True)

print(f"\nALL RESULTS, WEIGHTS, PLOTS SAVED TO:\n{final_folder}")



ALL RESULTS, WEIGHTS, PLOTS SAVED TO:
/content/drive/MyDrive/PCB_ResponseKD_Final_20251126_0530


In [12]:

# ========================================================
# 7. KEEP COLAB ALIVE (Optional)
# ========================================================
%%javascript
function ClickConnect(){
  console.log("Keeping Colab alive...");
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000);

<IPython.core.display.Javascript object>

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.9 MB/s eta 0:00:00
